# 4.3 · 测试与调试：Inspector 与日志

**Testing and debugging: Inspector and logs**

> **学完本课你能：** 在两个协议时代下用 Inspector（Web、CLI、TUI、mcpdo）调试 MCP 服务器，阅读宿主与服务器日志，修复典型的启动故障，并用自动化的契约测试和黄金样本测试把行为固定下来。

官方阅读材料：[MCP Inspector](https://modelcontextprotocol.io/docs/2026-07-28/tools/inspector) · [Inspector CLI client](https://modelcontextprotocol.io/docs/2026-07-28/tools/inspector/cli) · [Inspector protocol eras](https://modelcontextprotocol.io/docs/2026-07-28/tools/inspector/protocol-eras) · [Debugging](https://modelcontextprotocol.io/docs/2026-07-28/tools/debugging)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 你真正会敲的 Inspector 命令
Inspector 是一个通过 `npx` 获取的 Node 包，所以这个离线 Notebook 不会运行它。把这些命令放在终端旁边备用（把 `server.py` 换成你的入口点）：

```bash
# explore in the browser (prints a URL with a per-launch token)
npx @modelcontextprotocol/inspector uv run server.py

# one-shot CLI checks, in both protocol eras
npx @modelcontextprotocol/inspector --cli uv run server.py --protocol-era modern --method tools/list
npx @modelcontextprotocol/inspector --cli uv run server.py --protocol-era legacy --method tools/list
npx @modelcontextprotocol/inspector --cli uv run server.py --method tools/call \
    --tool-name notes_search --tool-arg query=mcp --format json | jq .result

# pass env and cwd to a stdio server; CI gate on schema portability
npx @modelcontextprotocol/inspector --cli -e NOTES_TOKEN=dev --cwd ~/notes uv run server.py --method tools/list --strict

# terminal UI, and a persistent connection
npx @modelcontextprotocol/inspector --tui uv run server.py
npx -p @modelcontextprotocol/inspector mcpdo connect @notes uv run server.py
```

下面的内容用 Python 重建了这些工作流的核心，让你能看清这些工具做了什么，并在自己的测试中复用这些思路。

### 2. 被测服务器
一个小型笔记服务器，包含工具（Tool）、一个资源（Resource）、一个资源模板（Resource Template）和一个提示模板（Prompt）。它<strong>相对于自己的源文件</strong>来解析数据文件，并把日志写到 <strong>stderr</strong>：这正是本课讲解的两个好习惯。

In [ ]:
import json
import os
import sys
import tempfile
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)
workdir = Path(tempfile.mkdtemp(prefix="inspect-"))
server_path = workdir / "notes_server.py"
(workdir / "notes.json").write_text(json.dumps({
    "mcp": "MCP is USB-C for AI applications.",
    "lsp": "The Language Server Protocol inspired MCP.",
    "eras": "The Inspector supports legacy, auto and modern protocol eras.",
}))
print("workdir:", workdir)

In [ ]:
%%writefile {server_path}
import json
import logging
import sys
from pathlib import Path
from typing import Annotated, Literal

from pydantic import Field

from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from mcp_types import ToolAnnotations

logging.basicConfig(level=logging.INFO, stream=sys.stderr, format="%(levelname)s %(name)s: %(message)s")
log = logging.getLogger("notes")
NOTES: dict[str, str] = json.loads((Path(__file__).parent / "notes.json").read_text())
log.info("loaded %d notes", len(NOTES))

mcp = MCPServer("notes", version="1.2.0")
RO = ToolAnnotations(readOnlyHint=True, openWorldHint=False)


@mcp.tool(annotations=RO)
def notes_search(query: Annotated[str, Field(description="Case-insensitive text to look for")],
                 match: Annotated[Literal["any", "all"], Field(description="Match any or all words")] = "any") -> list[str]:
    """Return the ids of notes whose text matches the query. Read one with notes_get."""
    words = query.lower().split()
    test = any if match == "any" else all
    return sorted(nid for nid, text in NOTES.items() if test(w in text.lower() for w in words))


@mcp.tool(annotations=RO)
def notes_get(note_id: Annotated[str, Field(description="A note id from notes_search")]) -> str:
    """Return the full text of one note."""
    if note_id not in NOTES:
        raise ToolError(f"Unknown note {note_id!r}. Call notes_search to list valid ids.")
    return NOTES[note_id]


@mcp.resource("notes://index")
def index() -> str:
    """All note ids, one per line."""
    return "\n".join(sorted(NOTES))


@mcp.resource("notes://{note_id}")
def note(note_id: str) -> str:
    """One note as a resource."""
    return NOTES[note_id]


@mcp.prompt()
def summarize(note_id: str) -> str:
    """Ask the model to summarise a note."""
    return f"Summarise note {note_id} in one sentence: {NOTES.get(note_id, '')}"


if __name__ == "__main__":
    mcp.run(transport="stdio")

### 3. 一个迷你 Inspector
Inspector CLI 会连接、运行一个方法，然后以有意义的退出码（Exit Code）退出。我们的 `inspect_server` 执行“列出所有内容，然后调用每个工具”这一遍检查。它<strong>根据每个 `inputSchema` 构造示例参数</strong>（第一个 enum 值、默认值，或基于类型的占位值），并返回一个 Inspector 风格的退出码：所有调用都成功时为 `0`，任何工具返回 `isError` 时为 `5`。

In [ ]:
import time

from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client


def sample_args(schema: dict, overrides: dict | None = None) -> dict:
    samples = {"string": "mcp", "integer": 1, "number": 1.0, "boolean": True, "array": [], "object": {}}
    args = {}
    for name, prop in schema.get("properties", {}).items():
        if name not in schema.get("required", []):
            continue                                    # like a form left at its defaults
        if "enum" in prop:
            args[name] = prop["enum"][0]
        else:
            args[name] = prop.get("default", samples.get(prop.get("type"), "mcp"))
    return args | (overrides or {})


async def inspect_server(target, overrides: dict | None = None) -> int:
    exit_code = 0
    async with Client(target) as c:
        caps = c.server_capabilities.model_dump(exclude_none=True)
        print(f"server: {c.server_info.name} {c.server_info.version} | protocol {c.protocol_version}")
        print("capabilities:", sorted(caps))
        tools = (await c.list_tools()).tools
        print("resources:", [str(r.uri) for r in (await c.list_resources()).resources],
              "| templates:", [t.uri_template for t in (await c.list_resource_templates()).resource_templates],
              "| prompts:", [p.name for p in (await c.list_prompts()).prompts])
        for tool in tools:
            args = sample_args(tool.input_schema, (overrides or {}).get(tool.name))
            started = time.perf_counter()
            r = await c.call_tool(tool.name, args)
            ms = (time.perf_counter() - started) * 1000
            status = "ERROR" if r.is_error else "ok"
            exit_code = 5 if r.is_error else exit_code
            hints = tool.annotations.model_dump(by_alias=True, exclude_none=True) if tool.annotations else {}
            print(f"  {status:5} {tool.name}({json.dumps(args)}) {ms:5.1f} ms  hints={hints}")
            shown = r.structured_content if r.structured_content is not None else r.content[0].text
            print(f"        -> {json.dumps(shown)[:90]}")
    return exit_code


params = StdioServerParameters(command=sys.executable, args=[str(server_path)])
with open(workdir / "notes.stderr.log", "w") as errlog:
    code = await inspect_server(stdio_client(params, errlog=errlog))
print("exit code:", code)

`notes_get("mcp")` 碰巧成功了，因为我们的占位字符串恰好是一个真实的 id。试试 `overrides={"notes_get": {"note_id": "nope"}}`，退出码就会变成 `5`，就像工具返回 `isError: true` 时的 `mcp-inspector --cli … --method tools/call` 一样。

### 4. 线路记录器：30 行实现 Protocol 标签页
Inspector 的 Protocol 标签页就是一份 JSON-RPC 消息的报文记录（线路报文记录（Wire Transcript））。我们在客户端和服务器之间放一个很小的 <strong>stdio 代理</strong>，也能得到同样的效果。它原样转发每一行，并把它追加到一个 JSONL 记录中，同时标记出 stdout 上任何不是 JSON 的内容。因为它只是另一条命令，所以能与任何 stdio 客户端配合使用，包括 Claude Desktop（`"command": "python", "args": ["proxy.py", "log.jsonl", "python", "server.py"]`）。

In [ ]:
proxy_path = workdir / "wire_proxy.py"
proxy_path.write_text('''
import json, subprocess, sys, threading, time
log_path, *cmd = sys.argv[1:]
child = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE)
lock = threading.Lock()

def record(direction, raw):
    text = raw.decode("utf-8", "replace").rstrip("\\n")
    try:
        entry = {"t": time.time(), "dir": direction, "msg": json.loads(text)}
    except json.JSONDecodeError:
        entry = {"t": time.time(), "dir": direction, "invalid": text}
    with lock, open(log_path, "a") as f:
        f.write(json.dumps(entry) + "\\n")

def client_to_server():
    for line in sys.stdin.buffer:
        record("->", line)
        child.stdin.write(line)
        child.stdin.flush()
    child.stdin.close()

threading.Thread(target=client_to_server, daemon=True).start()
for line in child.stdout:
    record("<-", line)
    sys.stdout.buffer.write(line)
    sys.stdout.buffer.flush()
sys.exit(child.wait())
''')


def show_transcript(path: Path, width: int = 150) -> list[dict]:
    entries = [json.loads(l) for l in path.read_text().splitlines()]
    for e in entries:
        if "invalid" in e:
            print(f"{e['dir']} !! NOT JSON-RPC: {e['invalid']!r}")
            continue
        m = e["msg"]
        kind = m.get("method") or ("result" if "result" in m else "error")
        body = m.get("params") or m.get("result") or m.get("error")
        print(f"{e['dir']} id={m.get('id', '-')!s:4} {kind:22} {json.dumps(body)[:width]}")
    return entries


transcript = workdir / "wire.jsonl"
recorded = StdioServerParameters(command=sys.executable,
                                 args=[str(proxy_path), str(transcript), sys.executable, str(server_path)])
with open(workdir / "notes.stderr.log", "w") as errlog:
    async with Client(stdio_client(recorded, errlog=errlog)) as c:
        await c.call_tool("notes_search", {"query": "protocol eras", "match": "all"})
        await c.call_tool("notes_get", {"note_id": "nope"})
entries = show_transcript(transcript)

从上到下阅读这份记录：客户端的 `server/discover` 探测，每个请求上的 2026-07-28 `_meta` 信封，SDK 客户端在第一次调用之后发出的一次 `tools/list`（用于获取工具的 `outputSchema` 并校验 `structuredContent`），以及针对未知笔记的 `isError: true` 结果——它是一个<em>结果</em>，而不是 JSON-RPC 错误。

In [ ]:
first_call = next(e["msg"] for e in entries if e["dir"] == "->" and e["msg"].get("method") == "tools/call")
print("one complete request, exactly as written to the server's stdin:")
print(json.dumps(first_call, indent=2))

### 5. 手动检查协议兼容性
调试指南的连接检查清单最后两项是“verify protocol compatibility”和“check the per-request `_meta` fields”。下面是你应当认得的三种应答：一个正常的 `server/discover`、一个缺少 `_meta` 的请求（`-32602`），以及一个不支持的版本（带 `data.supported` 的 `-32022`）。

In [ ]:
import subprocess

proc = subprocess.Popen([sys.executable, str(server_path)], text=True,
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0"}}


def rpc(id_, method, params):
    proc.stdin.write(json.dumps({"jsonrpc": "2.0", "id": id_, "method": method, "params": params}) + "\n")
    proc.stdin.flush()
    return json.loads(proc.stdout.readline())


ok = rpc(1, "server/discover", {"_meta": META})
print("server/discover ->", {k: ok["result"][k] for k in ("supportedVersions", "capabilities")})
print("missing _meta   ->", rpc(2, "tools/list", {})["error"])
old = dict(META, **{"io.modelcontextprotocol/protocolVersion": "2020-01-01"})
print("bad version     ->", rpc(3, "tools/list", {"_meta": old})["error"])
proc.stdin.close()
_ = proc.wait(timeout=10)

### 6. 测试套件：行为测试、契约测试和黄金线路测试
我们在服务器旁边写一个真正的 pytest 风格测试文件。它使用内存内的 `Client(server)` 以求速度，外加一个 stdio 冒烟测试。第一次运行会记录<strong>黄金文件</strong>（工具契约和一个线路格式的结果）。之后的运行都与它们进行比较（黄金样本测试（Golden Test））。如果安装了 `pytest`，就直接运行该文件；否则由一个 15 行的运行器执行同样的测试函数，并 await 其中的异步函数。

In [ ]:
tests_path = workdir / "test_notes_server.py"
tests_path.write_text('''
import importlib.util, json, os, sys
from pathlib import Path
import pytest
from mcp import Client, StdioServerParameters

HERE = Path(__file__).parent
GOLDEN = HERE / "golden"
spec = importlib.util.spec_from_file_location("notes_server", HERE / "notes_server.py")
notes_server = importlib.util.module_from_spec(spec)
spec.loader.exec_module(notes_server)
server = notes_server.mcp


def wire(model):
    data = model.model_dump(by_alias=True, exclude_none=True, mode="json")
    data.pop("_meta", None)                      # volatile/identity fields don't belong in goldens
    return data


def check_golden(name, value):
    path = GOLDEN / f"{name}.json"
    text = json.dumps(value, indent=2, sort_keys=True)
    if os.environ.get("UPDATE_GOLDEN") or not path.exists():
        GOLDEN.mkdir(exist_ok=True)
        path.write_text(text)
    if text != path.read_text():
        path.with_suffix(".actual.json").write_text(text)     # keep the new version for the diff
        raise AssertionError(f"{name} changed; review the diff, then re-run with UPDATE_GOLDEN=1")


@pytest.mark.anyio
async def test_search_finds_notes():
    async with Client(server) as c:
        r = await c.call_tool("notes_search", {"query": "protocol"})
    assert not r.is_error and r.structured_content == {"result": ["eras", "lsp"]}


@pytest.mark.anyio
async def test_unknown_note_is_actionable_error():
    async with Client(server) as c:
        r = await c.call_tool("notes_get", {"note_id": "nope"})
    assert r.is_error and "notes_search" in r.content[0].text


@pytest.mark.anyio
async def test_tool_contract():
    async with Client(server) as c:
        tools = [wire(t) for t in (await c.list_tools()).tools]
    check_golden("tools_contract", tools)


@pytest.mark.anyio
async def test_golden_wire_result():
    async with Client(server) as c:
        r = await c.call_tool("notes_search", {"query": "mcp"})
    check_golden("notes_search_mcp", wire(r))


@pytest.mark.anyio
async def test_stdio_smoke():
    params = StdioServerParameters(command=sys.executable, args=[str(HERE / "notes_server.py")], cwd="/")
    async with Client(params) as c:              # cwd="/" like a GUI host; catches path bugs
        assert {t.name for t in (await c.list_tools()).tools} == {"notes_search", "notes_get"}
''')

import difflib
import importlib.util
import inspect
import logging
import shutil

for name in ("notes", "mcp"):                       # keep server-side logs out of the test report
    logging.getLogger(name).setLevel(logging.WARNING)


HAVE_PYTEST = importlib.util.find_spec("pytest") is not None


async def run_tests(path: Path) -> bool:
    if HAVE_PYTEST:
        result = subprocess.run([sys.executable, "-m", "pytest", "-q", str(path)],
                                capture_output=True, text=True, cwd=path.parent)
        print(result.stdout[-1500:])
        return result.returncode == 0
    sys.modules.setdefault("pytest", type(sys)("pytest"))           # minimal stand-in for pytest.mark.anyio
    sys.modules["pytest"].mark = type("M", (), {"anyio": staticmethod(lambda f: f)})
    spec = importlib.util.spec_from_file_location("test_notes_server", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    passed = True
    for name, fn in inspect.getmembers(module, inspect.isfunction):
        if not name.startswith("test_"):
            continue
        try:
            await fn() if inspect.iscoroutinefunction(fn) else fn()
            print(f"PASS {name}")
        except Exception as exc:
            passed = False
            print(f"FAIL {name}: {type(exc).__name__}: {str(exc)[:300]}")
    return passed


shutil.rmtree(workdir / "golden", ignore_errors=True)
print("run 1 (records goldens):", await run_tests(tests_path))
print("goldens:", sorted(p.name for p in (workdir / "golden").iterdir()))
print("\nrun 2 (compares):", await run_tests(tests_path))

现在做一个看似无害的修改，把参数 `match` 重命名为 `mode`，然后再跑一遍测试套件。行为测试仍然通过，但<strong>契约测试（Contract Test）</strong>捕捉到了一个对所有使用 `match` 的客户端和提示词都具有破坏性的变更。

In [ ]:
original_source = server_path.read_text()
server_path.write_text(original_source.replace("match: Annotated", "mode: Annotated")
                       .replace('if match == "any"', 'if mode == "any"'))
print("after renaming a parameter:", await run_tests(tests_path))
golden = workdir / "golden" / "tools_contract.json"
print("".join(l for l in difflib.unified_diff(golden.read_text().splitlines(True),
                                               golden.with_suffix(".actual.json").read_text().splitlines(True),
                                               "golden", "actual", n=0) if not l.startswith("@@")))
_ = server_path.write_text(original_source)           # revert

### 7. 一步步调试一个坏掉的服务器
这是一个“在我机器上能跑”的服务器：在终端里从它自己的文件夹运行时一切正常。但宿主（Host）启动它的方式不同：脚本使用绝对路径，工作目录（Working Directory）是 `/`，环境变量也极少。我们像宿主那样启动它，像读 `mcp-server-NAME.log` 那样读它的 stderr 日志，每一步修复一个 bug。

In [ ]:
broken_path = workdir / "broken_notes.py"
broken_path.write_text('''
import json, os
from mcp.server import MCPServer

NOTES = json.load(open("notes.json"))           # bug 1: relative to the *working directory*
TOKEN = os.environ["NOTES_TOKEN"]               # bug 2: env var the host does not pass
print(f"Loaded {len(NOTES)} notes", flush=True) # bug 3: stdout is the wire

mcp = MCPServer("broken-notes")

@mcp.tool()
def count_notes() -> int:
    """How many notes exist."""
    return len(NOTES)

if __name__ == "__main__":
    mcp.run()
''')


def leaf_error(exc: BaseException) -> BaseException:
    while isinstance(exc, BaseExceptionGroup):
        exc = exc.exceptions[0]
    return exc


async def launch_like_a_host(env: dict | None = None, record_to: Path | None = None) -> None:
    log_path = workdir / "mcp-server-broken-notes.log"
    command = [str(broken_path)]
    if record_to:
        command = [str(proxy_path), str(record_to), sys.executable, str(broken_path)]
    params = StdioServerParameters(command=sys.executable, args=command, cwd="/", env=env)
    try:
        with open(log_path, "w") as errlog:
            async with Client(stdio_client(params, errlog=errlog)) as c:
                r = await c.call_tool("count_notes", {})
                print("CONNECTED: count_notes ->", r.structured_content)
    except Exception as exc:
        print("CLIENT SAW:", repr(leaf_error(exc)))
        tail = log_path.read_text().strip().splitlines()[-1:]
        print("LOG TAIL  :", tail)


print("step 1: launch it like a host would")
await launch_like_a_host()

客户端只看到 `Connection closed`，这也是宿主能告诉你的全部信息。<strong>日志</strong>里才有原因：`notes.json` 是在 `/` 中查找的。修复 1：相对于源文件来解析数据。

In [ ]:
src = broken_path.read_text()
src = src.replace('json.load(open("notes.json"))',
                  'json.load(open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "notes.json")))')
broken_path.write_text(src)
print("step 2: after fixing the path")
await launch_like_a_host()

新的故障，新的日志行：`KeyError: 'NOTES_TOKEN'`。stdio 服务器只继承有限的环境变量，所以要显式传入，这相当于 `claude_desktop_config.json` 中的一个 `"env"` 条目。

In [ ]:
print("step 3: pass the environment explicitly, and record the wire")
wire_log = workdir / "broken-wire.jsonl"
await launch_like_a_host(env={"NOTES_TOKEN": "dev-token"}, record_to=wire_log)
_ = show_transcript(wire_log, width=90)

它连上了，因为 Python 客户端能容忍那行垃圾内容，但记录器暴露出了 <strong>bug 3</strong>：在第一个响应（Response）之前，stdout 上出现了一行非 JSON 内容。更严格的宿主可能会在这里断开连接。修复 3：把日志写到 stderr。然后确认报文记录是干净的。

In [ ]:
broken_path.write_text(broken_path.read_text().replace(
    'print(f"Loaded {len(NOTES)} notes", flush=True)', 'print(f"Loaded {len(NOTES)} notes", file=sys.stderr)')
    .replace("import json, os", "import json, os, sys"))
wire_log.unlink()
print("step 4: after logging to stderr")
await launch_like_a_host(env={"NOTES_TOKEN": "dev-token"}, record_to=wire_log)
bad_lines = [e for e in map(json.loads, wire_log.read_text().splitlines()) if "invalid" in e]
print("non-JSON lines on the wire:", bad_lines)
print("server log:", (workdir / "mcp-server-broken-notes.log").read_text().strip())

每个 bug 都只有在<strong>复现宿主的启动条件</strong>并<strong>读对日志</strong>时才会显现出来。把每一次修复都变成一个回归测试：第 6 节的 `test_stdio_smoke` 已经在用 `cwd="/"` 启动了。

### 动手练习
1. 给 `notes_server.py` 添加第三个工具（比如 `notes_count`），然后重新运行第 6 节。哪个测试失败了？用什么命令（环境变量）可以接受新的契约？
2. 扩展 `inspect_server`，让它读取每一个静态资源，并用示例参数渲染每一个提示模板，按报告工具的方式报告失败。
3. 通过加载服务器模块，在进程内用 `Client(..., mode="legacy")` 运行 `inspect_server`。它报告的是哪个协议版本？还有其他变化吗？
4. 让线路记录器也测量延迟：按 `id` 把每个 `->` 请求与对应的 `<-` 响应配对，并打印每个方法耗费的毫秒数。